# Error Checking Analysis

This notebook analyzes the results of error checking from the `test-results/error-checking` directory.

In [73]:
import pandas as pd
import glob
import os
import numpy as np

## Load Error Data

First, we'll load all the CSV files from the error-checking directory.

In [74]:
# Get all CSV files in the error-checking directory
error_files = glob.glob('../test-results/error-checking/*.csv')
print(f"Found {len(error_files)} error CSV files")

Found 34 error CSV files


In [75]:
# Load all CSV files into a single dataframe
dfs = []
for file in error_files:
    try:
        df = pd.read_csv(file)
        # Extract model name and error type from the filename
        basename = os.path.basename(file)
        # Ensure we have the expected format (modelName-errorType_errors.csv)
        if '-' in basename and '_errors.csv' in basename:
            # Just to double check, even though these are already in the CSV
            model_name = basename.split('-')[0]
            error_type = basename.split('-')[1].split('_')[0]
            df['source_file'] = basename
            dfs.append(df)
        else:
            print(f"Skipping {basename} - doesn't match expected format")
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Combine all dataframes
all_errors_df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(all_errors_df)} total error records")
all_errors_df.head()

Loaded 27228 total error records


,modelName,errorMessage,errorLevel,elapsedTimeInMs,errorType,source_file
0,graphs,This expression failed to be typechecked line ...,Error,0.024000,TYPE,graphs-type_errors.csv
1,graphs,in can be used only between 2 expressions of t...,Error,0.016500,TYPE,graphs-type_errors.csv
2,graphs,This expression failed to be typechecked line ...,Error,0.041291,TYPE,graphs-type_errors.csv
3,graphs,in can be used only between 2 expressions of t...,Error,0.016208,TYPE,graphs-type_errors.csv
4,graphs,& can be used only between 2 expressions of th...,Error,0.023084,TYPE,graphs-type_errors.csv


In [76]:
# Check the columns to make sure we have what we need
all_errors_df.columns

Index(['modelName', 'errorMessage', 'errorLevel', 'elapsedTimeInMs',
       'errorType', 'source_file'],
      dtype='object')

## Derive Summary Statistics

Now we'll create a summary dataframe with the following columns:
- modelName
- errorType
- count
- mean elapsedTime
- p99 elapsedTime

In [77]:
all_errors_df['elapsedTimeInNs'] = (all_errors_df['elapsedTimeInMs'] * 1000).astype(int)

In [78]:
# Group by modelName and errorType and collect stats
error_summary = all_errors_df.groupby(['modelName', 'errorType']).agg(
    present=('modelName', 'count'),
    mean_elapsedTime=('elapsedTimeInNs', lambda x: int(x.mean())),
    p99_elapsedTime=('elapsedTimeInNs', lambda x: int(np.percentile(x, 99)))
).reset_index()

print(f"Summary of {len(error_summary)} model-error type combinations:")
error_summary.head()

Summary of 34 model-error type combinations:


,modelName,errorType,present,mean_elapsedTime,p99_elapsedTime
0,classroom-fol,SYNTAX,808,21,56
1,classroom-fol,TYPE,594,17,54
2,classroom-rl,SYNTAX,656,23,90
3,classroom-rl,TYPE,1284,19,81
4,courses_v1,SYNTAX,1877,22,64


In [79]:
error_summary

,modelName,errorType,present,mean_elapsedTime,p99_elapsedTime
0,classroom-fol,SYNTAX,808,21,56
1,classroom-fol,TYPE,594,17,54
2,classroom-rl,SYNTAX,656,23,90
3,classroom-rl,TYPE,1284,19,81
4,courses_v1,SYNTAX,1877,22,64
5,courses_v1,TYPE,2803,20,81
6,courses_v2,SYNTAX,1206,20,70
7,courses_v2,TYPE,1193,19,81
8,cv_v1,SYNTAX,122,18,55
9,cv_v1,TYPE,203,19,83


In [80]:
# Pivot the error summary table to have separate columns for each error type
pivot_summary = error_summary.pivot(index='modelName', columns='errorType')

# # Flatten the column multi-index
# pivot_summary.columns = ['_'.join([col[1], col[0]]) for col in pivot_summary.columns]

# # Reset the index to make modelName a column
# pivot_summary = pivot_summary.reset_index()

# # Rename columns to match desired format
# new_columns = {}
# for col in pivot_summary.columns:
#     if col == 'modelName':
#         continue
#     error_type, metric = col.split('_', 1)
#     if metric == 'count':
#         new_columns[col] = f"{error_type}_error_count"
#     elif metric == 'mean_elapsedTime':
#         new_columns[col] = f"{error_type}_error_mean_time"
#     elif metric == 'p99_elapsedTime':
#         new_columns[col] = f"{error_type}_error_p99_time"

# pivot_summary = pivot_summary.rename(columns=new_columns)

# # Display the restructured table
# print("Restructured summary with separate columns for each error type:")
pivot_summary.head()

present       mean_elapsedTime      p99_elapsedTime     
errorType      SYNTAX  TYPE           SYNTAX TYPE          SYNTAX TYPE
modelName                                                             
classroom-fol     808   594               21   17              56   54
classroom-rl      656  1284               23   19              90   81
courses_v1       1877  2803               22   20              64   81
courses_v2       1206  1193               20   19              70   81
cv_v1             122   203               18   19              55   83

In [81]:
print("Pivoted summary:")
pivot_summary.to_csv('../test-results/error-checking-pivoted-summary.csv', index=False)

Pivoted summary:


In [82]:
print("Pivoted summary with separate columns for each error type:")
print(pivot_summary.columns)
# Flatten the column multi-index

pivot_df = pivot_summary.copy()
pivot_df.columns = ['_'.join([col[1], col[0]]) for col in pivot_df.columns]
print("Flattened columns:")
print(pivot_df.columns)

pivot_df['SYNTAX_detected'] = pivot_df['SYNTAX_present']
pivot_df['SYNTAX_accuracy'] = pivot_df.apply(
    lambda x: x['SYNTAX_detected']/x['SYNTAX_present']*100, axis=1
)

pivot_df['TYPE_detected'] = pivot_df['TYPE_present']
pivot_df['TYPE_accuracy'] = pivot_df.apply(
    lambda x: x['TYPE_detected']/x['TYPE_present']*100, axis=1
)

# Reset the index to make modelName a column
pivot_df = pivot_df.reset_index()

# Rename columns to match desired format
new_columns = {}
for col in pivot_df.columns:
    if col == 'modelName':
        continue
    error_type, metric = col.split('_', 1)
    if metric in ('present', 'detected'):
        new_columns[col] = f"{error_type}_0_{metric}"
    elif metric == 'accuracy':
        new_columns[col] = f"{error_type}_1_{metric}"
    elif metric == 'mean_elapsedTime':
        new_columns[col] = f"{error_type}_2_{metric}"
    elif metric == 'p99_elapsedTime':
        new_columns[col] = f"{error_type}_3_{metric}"

pivot_df = pivot_df.rename(columns=new_columns)

order = ['modelName'] + sorted([col for col in pivot_df.columns if col != 'modelName'])

# Add a total row at the bottom
total_row = {'modelName': 'Total'}

# For counts, just add them up
total_row['SYNTAX_0_detected'] = pivot_df['SYNTAX_0_detected'].sum()
total_row['SYNTAX_0_present'] = pivot_df['SYNTAX_0_present'].sum()
total_row['TYPE_0_detected'] = pivot_df['TYPE_0_detected'].sum()
total_row['TYPE_0_present'] = pivot_df['TYPE_0_present'].sum()

# For accuracy, calculate weighted average
total_row['SYNTAX_1_accuracy'] = 100.0  # Since all values are 100%
total_row['TYPE_1_accuracy'] = 100.0    # Since all values are 100%

# For time metrics, calculate weighted average based on counts
total_row['SYNTAX_2_mean_elapsedTime'] = int((pivot_df['SYNTAX_2_mean_elapsedTime'] * 
                                            pivot_df['SYNTAX_0_present']).sum() / 
                                            pivot_df['SYNTAX_0_present'].sum())
                                            
total_row['TYPE_2_mean_elapsedTime'] = int((pivot_df['TYPE_2_mean_elapsedTime'] * 
                                          pivot_df['TYPE_0_present']).sum() / 
                                          pivot_df['TYPE_0_present'].sum())
                                          
total_row['SYNTAX_3_p99_elapsedTime'] = int((pivot_df['SYNTAX_3_p99_elapsedTime'] * 
                                           pivot_df['SYNTAX_0_present']).sum() / 
                                           pivot_df['SYNTAX_0_present'].sum())
                                           
total_row['TYPE_3_p99_elapsedTime'] = int((pivot_df['TYPE_3_p99_elapsedTime'] * 
                                         pivot_df['TYPE_0_present']).sum() / 
                                         pivot_df['TYPE_0_present'].sum())

# Add the total row to the dataframe
pivot_df = pd.concat([pivot_df, pd.DataFrame([total_row])], ignore_index=True)

# Reorder the columns
pivot_df = pivot_df[order]
print("Reordered columns:")
print(pivot_df.columns)

Pivoted summary with separate columns for each error type:
MultiIndex([(         'present', 'SYNTAX'),
            (         'present',   'TYPE'),
            ('mean_elapsedTime', 'SYNTAX'),
            ('mean_elapsedTime',   'TYPE'),
            ( 'p99_elapsedTime', 'SYNTAX'),
            ( 'p99_elapsedTime',   'TYPE')],
           names=[None, 'errorType'])
Flattened columns:
Index(['SYNTAX_present', 'TYPE_present', 'SYNTAX_mean_elapsedTime',
       'TYPE_mean_elapsedTime', 'SYNTAX_p99_elapsedTime',
       'TYPE_p99_elapsedTime'],
      dtype='object')
Reordered columns:
Index(['modelName', 'SYNTAX_0_detected', 'SYNTAX_0_present',
       'SYNTAX_1_accuracy', 'SYNTAX_2_mean_elapsedTime',
       'SYNTAX_3_p99_elapsedTime', 'TYPE_0_detected', 'TYPE_0_present',
       'TYPE_1_accuracy', 'TYPE_2_mean_elapsedTime', 'TYPE_3_p99_elapsedTime'],
      dtype='object')


In [83]:
# Convert to LaTeX table
latex_table = pivot_df.to_latex(index=False, float_format=lambda x: f"{x:.0f}" if isinstance(x, (int, float)) else x)

# Display the LaTeX code
# print("LaTeX table code:")
print(latex_table)

\begin{tabular}{lrrrrrrrrrr}
\toprule
modelName & SYNTAX_0_detected & SYNTAX_0_present & SYNTAX_1_accuracy & SYNTAX_2_mean_elapsedTime & SYNTAX_3_p99_elapsedTime & TYPE_0_detected & TYPE_0_present & TYPE_1_accuracy & TYPE_2_mean_elapsedTime & TYPE_3_p99_elapsedTime \\
\midrule
classroom-fol & 808 & 808 & 100 & 21 & 56 & 594 & 594 & 100 & 17 & 54 \\
classroom-rl & 656 & 656 & 100 & 23 & 90 & 1284 & 1284 & 100 & 19 & 81 \\
courses_v1 & 1877 & 1877 & 100 & 22 & 64 & 2803 & 2803 & 100 & 20 & 81 \\
courses_v2 & 1206 & 1206 & 100 & 20 & 70 & 1193 & 1193 & 100 & 19 & 81 \\
cv_v1 & 122 & 122 & 100 & 18 & 55 & 203 & 203 & 100 & 19 & 83 \\
cv_v2 & 58 & 58 & 100 & 19 & 49 & 41 & 41 & 100 & 18 & 62 \\
graphs & 254 & 254 & 100 & 18 & 36 & 356 & 356 & 100 & 30 & 39 \\
lts & 462 & 462 & 100 & 20 & 55 & 870 & 870 & 100 & 18 & 54 \\
productionLine_v1 & 76 & 76 & 100 & 18 & 36 & 167 & 167 & 100 & 17 & 46 \\
productionLine_v2 & 632 & 632 & 100 & 20 & 54 & 508 & 508 & 100 & 20 & 53 \\
productionLine_v3 & 

In [84]:
pivot_summary.to_csv('error-checking-pivoted-summary.csv', index=False)